In [0]:
# Installation des librairies de base
%pip install langgraph langchain-community langchain-huggingface faiss-cpu sentence-transformers

# Mise à jour forcée pour éviter le conflit Pydantic/Typing (l'erreur de tout à l'heure)
%pip install -U typing_extensions pydantic langchain-core

# Redémarrage du moteur pour prendre en compte les installations
dbutils.library.restartPython()

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import os

sys.path.append(os.path.abspath('..'))
from src.interview_graph import build_interview_graph

# --- LA CORRECTION EST ICI ---
# 2. On récupère le token secret depuis Databricks
try:
    hf_token = dbutils.secrets.get(scope="llm_secrets", key="huggingface_token")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
    print(" Token HuggingFace récupéré et injecté dans l'environnement !")
except Exception as e:
    print(f" Erreur avec le secret Databricks : {e}")

# Les listes qui simulent l'entretien complet
liste_questions = [
    "Qu'est-ce qu'une clé primaire en SQL ?",
    "C'est quoi un index en SQL ?",
    "Comment optimiser une requête lente ?"
]

liste_reponses = [
    "C'est un identifiant unique pour chaque ligne.",                      # Bonne
    "Ça ralentit la base de données, c'est pas très utile.",               # Mauvaise
    "On peut ajouter des index sur les colonnes WHERE et éviter SELECT *." # Bonne
]

etat_initial = {
    "questions": liste_questions,
    "reponses": liste_reponses,
    "contextes_rag": [],
    "feedback_recruteur": "",
    "passed_recruteur": False,
    "verdict_juge": "",
    "juge_est_daccord": True
}

print(" Lancement de l'analyse globale de l'entretien...\n")

app = build_interview_graph()
resultat = app.invoke(etat_initial)

print("\n================ RÉSULTAT FINAL ================")
print("\n [BILAN DU RECRUTEUR]")
print(resultat["feedback_recruteur"])

print("\n DÉCISION FINALE (Passed) :", resultat["passed_recruteur"])

print("\n [AVIS DU DIRECTEUR RH (JUGE)]")
print(resultat["verdict_juge"])